# Tutorial

In [1]:
import numpy as np

# Generar el array aleatorio con números flotantes entre 0 y 1 (distribución uniforme)
array_aleatorio = np.array([
                            [
                                [0.37596648, 0.07939675],
                                [0.0,        0.52823864]
                            ],
                            [
                                [0.23701405, 0.29549536],
                                [0.40713081, 0.0],
                            ]
    
])
array_aleatorio[0][1][0] = np.nan
array_aleatorio[1][1][1] = np.nan

In [2]:
array_aleatorio

array([[[0.37596648, 0.07939675],
        [       nan, 0.52823864]],

       [[0.23701405, 0.29549536],
        [0.40713081,        nan]]])

In [3]:
array_artificialmente_eliminado = array_aleatorio.copy()
array_artificialmente_eliminado[0][1][1] = np.nan
array_artificialmente_eliminado

array([[[0.37596648, 0.07939675],
        [       nan,        nan]],

       [[0.23701405, 0.29549536],
        [0.40713081,        nan]]])

In [4]:
# Máscara CORRECTA Y PRECISA para evaluar la imputación:
# 1 sólo si fue artificialmente eliminado Y el original NO era NaN
mask_artificialmente_nan = np.isnan(array_artificialmente_eliminado)
mask_original_presente = ~np.isnan(array_aleatorio)
corrected_evaluation_mask = (mask_artificialmente_nan & mask_original_presente).astype(int)

print("Máscara de Evaluación CORREGIDA:\n", corrected_evaluation_mask)
array_artificialmente_eliminado

Máscara de Evaluación CORREGIDA:
 [[[0 0]
  [0 1]]

 [[0 0]
  [0 0]]]


array([[[0.37596648, 0.07939675],
        [       nan,        nan]],

       [[0.23701405, 0.29549536],
        [0.40713081,        nan]]])

In [5]:
def impute_median_sample_wise(data, fallback_value=0.0):
    """
    Imputa los valores faltantes usando la mediana de cada característica dentro de cada muestra
    (calculada a través del eje de tiempo), usando un enfoque vectorizado.

    Si toda una característica para una muestra es NaN, usa un valor de reserva (`fallback_value`).

    Args:
        data (ndarray): Array NumPy de forma (n_samples, n_timestamps, n_features)
                        Datos de entrada con valores NaN para ser imputados.
        fallback_value (float, optional): Valor a usar si toda una característica
                                          dentro de una muestra es NaN. Por defecto es 0.0.

    Returns:
        ndarray: Array NumPy de forma (n_samples, n_timestamps, n_features)
                 Datos con valores NaN imputados.
    """
    # Asegurarse de que la entrada es un array NumPy
    data = np.asarray(data)
    if data.ndim != 3:
        raise ValueError(f"Input data must be 3-dimensional (samples, timestamps, features), but got {data.ndim} dimensions.")
    medians = np.nanmedian(data, axis=(1,2), keepdims=True)

    medians = np.where(np.isnan(medians), fallback_value, medians)
    nan_mask = np.isnan(data)

    imputed_data = np.where(nan_mask, medians, data)

    return imputed_data

In [6]:
array_imputado = impute_median_sample_wise(array_artificialmente_eliminado)

In [7]:
array_imputado

array([[[0.37596648, 0.07939675],
        [0.22768162, 0.22768162]],

       [[0.23701405, 0.29549536],
        [0.40713081, 0.29549536]]])

In [8]:
from pypots.nn.functional import calc_mae

predictions = array_imputado
targets = np.nan_to_num(array_aleatorio, nan=0.0)
mask = corrected_evaluation_mask

print(calc_mae(predictions, targets, mask))


████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 

0.3005570249996994
